# Example queries: `helpers` (comstock_oedi_agg)

Auto-generated from `tests/query_snapshots/helpers.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [1]:
from pathlib import Path
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [2]:
# This notebook lives in `tests/example_notebooks/`; the snapshot test
# cache is its sibling `tests/query_snapshots/comstock_oedi_agg_cache/`. Resolve
# the path relative to the notebook directory (`_dh[0]` is set by
# IPython at kernel startup; falls back to CWD outside Jupyter).
_NB_DIR = Path(_dh[0] if "_dh" in globals() else ".").resolve()
_CACHE = (_NB_DIR / "../query_snapshots/comstock_oedi_agg_cache").resolve()
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "comstock_amy2018_r2_2025",
    buildstock_type="comstock",
    db_schema="comstock_oedi_agg_state_and_county",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


INFO:buildstock_query.query_core:Loading comstock_amy2018_r2_2025 ...


INFO:botocore.tokens:Loading cached SSO token for nrel-sso


## `upgrade_names_oedi`

get_upgrade_names — was broken before 2026-04-26 fix on two counts: (1) f-string interpolation of self.up_table (a Subquery) produced malformed `FROM SELECT * FROM ...` SQL with no enclosing parens; (2) the method assumed an `apply_upgrade.upgrade_name` column that doesn't exist on OEDI schemas. Fixed to use SA construction and gracefully degrade to `CAST(NULL AS VARCHAR) AS upgrade_name` when the name column is absent — so the dict shape stays consistent across schemas.


In [3]:
result = bsq.get_upgrade_names()
result.head() if hasattr(result, 'head') else result


{1: 'Variable Speed HP RTU, Electric Backup',
 2: 'Variable Speed HP RTU, Original Heating Fuel Backup',
 3: 'Variable Speed HP RTU, Electric Backup, Energy Recovery',
 4: 'Standard Performance HP RTU, Electric Backup',
 5: 'Standard Performance HP RTU using Lab Data, Electric Backup',
 6: 'Standard Performance HP RTU, Electric Backup + Roof Insulation',
 7: 'Standard Performance HP RTU, Electric Backup + New Windows',
 8: 'Standard Performance HP RTU, Electric Backup, 32F Minimum Compressor Lockout',
 9: 'Standard Performance HP RTU, Electric Backup, 2F Unoccupied Htg Thermostat Setback',
 10: 'Cold Climate Challenge HP RTU, Electric Backup',
 11: 'VRF with DOAS',
 12: 'VRF with 25pct Upsizing Allowance',
 13: 'DOAS HP Minisplits',
 14: 'HP Boiler, Electric Backup',
 15: 'HP Boiler, Gas Backup',
 16: 'Condensing Gas Boilers',
 17: 'Electric Resistance Boilers',
 18: 'Air Side Economizers for AHUs',
 19: 'Demand Control Ventilation',
 20: 'Energy Recovery for AHUs',
 21: 'Advanced RTU 

## `upgrades_csv_three_buildings_upgrade1`

get_upgrades_csv for upgrade=1 restricted to a tiny known building_id list. Mirrors results_csv_three_buildings on the upgrade-table side.


In [4]:
result = bsq.get_upgrades_csv(upgrade_id='1', restrict=[('bldg_id', [1, 2, 3])])
result.head() if hasattr(result, 'head') else result


INFO:buildstock_query.main:Making results_csv query for upgrade ...


,,,upgrade,weight,in.sqft..ft2,calc.weighted.sqft..ft2,in.upgrade_name,applicability,completed_status,dataset,applicability.add_console_gshp,applicability.add_heat_pump_rtu,...,calc.weighted.utility_bills.electricity_fixedcharge_bill_savings_median_low..billion_usd,calc.weighted.utility_bills.electricity_fixedcharge_bill_savings_min..billion_usd,calc.weighted.utility_bills.fuel_oil_bill_savings_state_average..billion_usd,calc.weighted.utility_bills.fuel_oil_bill_state_average..billion_usd,calc.weighted.utility_bills.natural_gas_bill_savings_state_average..billion_usd,calc.weighted.utility_bills.natural_gas_bill_state_average..billion_usd,calc.weighted.utility_bills.propane_bill_savings_state_average..billion_usd,calc.weighted.utility_bills.propane_bill_state_average..billion_usd,calc.weighted.utility_bills.total_bill_mean..billion_usd,calc.weighted.utility_bills.total_bill_savings_mean..billion_usd
bldg_id,county,state,,,,,,,,,,,,,,,,,,,,,
1,G0100710,AL,1,1.541059,1000.0,1541.059280,"Variable Speed HP RTU, Electric Backup",True,Success,ComStock sdr_2025_r3_combined,False,True,...,0.000000e+00,0.000000e+00,0.0,0.0,0.0,5.791301e-06,0.0,0.0,0.000040,1.254047e-06
2,G0100710,AL,1,0.352777,2000.0,705.554818,"Variable Speed HP RTU, Electric Backup",True,Success,ComStock sdr_2025_r3_combined,False,True,...,0.000000e+00,0.000000e+00,0.0,0.0,0.0,4.864800e-07,0.0,0.0,0.000009,1.581148e-06
3,G0100710,AL,1,0.203549,5500.0,1119.520173,"Variable Speed HP RTU, Electric Backup",True,Success,ComStock sdr_2025_r3_combined,False,True,...,0.000000e+00,0.000000e+00,0.0,0.0,0.0,8.376046e-07,0.0,0.0,0.000011,1.564461e-06
1,G0100350,AL,1,0.398619,1000.0,398.618541,"Variable Speed HP RTU, Electric Backup",True,Success,ComStock sdr_2025_r3_combined,False,True,...,3.025515e-07,3.025515e-07,0.0,0.0,0.0,1.498008e-06,0.0,0.0,0.000009,3.025515e-07
2,G0100350,AL,1,0.199309,2000.0,398.618541,"Variable Speed HP RTU, Electric Backup",True,Success,ComStock sdr_2025_r3_combined,False,True,...,1.014285e-06,1.014285e-06,0.0,0.0,0.0,2.748475e-07,0.0,0.0,0.000005,1.014285e-06


## `buildings_by_locations_state`

get_buildings_by_locations: list bs-keys whose location_col=$STATE_COL_BASELINE is in ['CO']. Different code path from restrict — uses bs_key_cols projection and IN-list filter directly.


In [5]:
result = bsq.get_buildings_by_locations(location_col='state', locations=['CO'])
result.head() if hasattr(result, 'head') else result


,bldg_id,county,state
0,46369,G0800150,CO
1,46369,G0800190,CO
2,46369,G0800370,CO
3,46369,G0800490,CO
4,46369,G0800510,CO


## `results_csv_three_buildings`

get_results_csv restricted to a tiny known building_id list. Pins the SELECT-* shape of the baseline metadata projection used by basic_usage.ipynb. Bounded to 3 buildings so the data parquet stays small (results_csv has ~150 columns, full state would be 100+ MB).


In [6]:
result = bsq.get_results_csv(restrict=[('bldg_id', [1, 2, 3])])
result.head() if hasattr(result, 'head') else result


INFO:buildstock_query.main:Making results_csv query ...


upgrade    weight  in.sqft..ft2  \
bldg_id county   state                                    
1       G2800410 MS           0  1.209209        1000.0   
        G2800530 MS           0  0.331651        1000.0   
        G2801050 MS           0  5.108097        1000.0   
2       G2801050 MS           0  3.039666        2000.0   
3       G2801050 MS           0  0.952988        5500.0   

                        calc.weighted.sqft..ft2 in.upgrade_name  \
bldg_id county   state                                            
1       G2800410 MS                 1209.209344        Baseline   
        G2800530 MS                  331.650626        Baseline   
        G2801050 MS                 5108.097295        Baseline   
2       G2801050 MS                 6079.331370        Baseline   
3       G2801050 MS                 5241.435197        Baseline   

                        applicability completed_status  \
bldg_id county   state                                   
1       G2800410 MS              True          Success   
        G2800530 MS              True          Success   
        G2801050 MS              True          Success   
2       G2801050 MS              True          Success   
3       G2801050 MS              True          Success   

                                              dataset  \
bldg_id county   state                                  
1       G2800410 MS     ComStock sdr_2025_r3_combined   
        G2800530 MS     ComStock sdr_2025_r3_combined   
        G2801050 MS     ComStock sdr_2025_r3_combined   
2       G2801050 MS     ComStock sdr_2025_r3_combined   
3       G2801050 MS     ComStock sdr_2025_r3_combined   

                        applicability.add_console_gshp  \
bldg_id county   state                                   
1       G2800410 MS                              False   
        G2800530 MS                              False   
        G2801050 MS                              False   
2       G2801050 MS                              False   
3       G2801050 MS                              False   

                        applicability.add_heat_pump_rtu  ...  \
bldg_id county   state                                   ...   
1       G2800410 MS                               False  ...   
        G2800530 MS                               False  ...   
        G2801050 MS                               False  ...   
2       G2801050 MS                               False  ...   
3       G2801050 MS                               False  ...   

                        calc.weighted.utility_bills.electricity_fixedcharge_bill_savings_median_low..billion_usd  \
bldg_id county   state                                                                                             
1       G2800410 MS                                                   0.0                                          
        G2800530 MS                                                   0.0                                          
        G2801050 MS                                                   0.0                                          
2       G2801050 MS                                                   0.0                                          
3       G2801050 MS                                                   0.0                                          

                        calc.weighted.utility_bills.electricity_fixedcharge_bill_savings_min..billion_usd  \
bldg_id county   state                                                                                      
1       G2800410 MS                                                   0.0                                   
        G2800530 MS                                                   0.0                                   
        G2801050 MS                                                   0.0                                   
2       G2801050 MS                                                   0.0                                   
3       G2

## `distinct_vals_state_baseline`

get_distinct_vals on the baseline-table state column. Pins the simple SELECT DISTINCT shape — used by notebooks to enumerate categorical values before building restricts. The column name differs by schema (resstock=`in.state`, comstock=`state`); resolved via $STATE_COL_BASELINE.


In [7]:
result = bsq.get_distinct_vals(
    column='state',
    table_name='comstock_amy2018_r2_2025_md_agg_by_state_and_county_parquet',
)
result.head() if hasattr(result, 'head') else result


0    AZ
1    PA
2    DC
3    TX
4    DE
Name: state, dtype: str

## `distinct_count_state_baseline`

get_distinct_count on the baseline-table state column. Pins the COUNT-with-weighted-sum shape (sample_count + weighted_count per category).


In [8]:
result = bsq.get_distinct_count(column='state')
result.head() if hasattr(result, 'head') else result


,state,sample_count,weighted_count
0,AK,4682,8113.081188
1,AL,36385,98577.243244
2,AR,28730,58124.142056
3,AZ,11225,86181.239323
4,CA,69450,449744.576018
